In [9]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI

In [5]:
# Kleine Fake-Firmadokumente
docs = [
    "LAIKA wurde 2024 gegründet. CEO ist Anna Müller. Hauptprodukt ist ein KI-Assistent für Firmen.",
    "Das Büro ist in Berlin. Wir haben 42 Mitarbeiter. Hunde sind im Büro erlaubt.",
    "Urlaubsantrag: Mindestens 2 Wochen vorher einreichen. Genehmigung durch Teamlead.",
    "Tech Stack: Python, PostgreSQL, Docker, Kubernetes. Kein PHP. Niemals.",
    "Onboarding dauert 3 Tage. Laptop-Setup, Team-Vorstellung, erstes Ticket am Tag 3.",
]
print(f"{len(docs)} Dokumente geladen ✅")

5 Dokumente geladen ✅


In [6]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Chroma = Vektordatenbank, läuft in-memory
vectorstore = Chroma.from_texts(docs, embeddings)
print("Vektordatenbank erstellt ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vektordatenbank erstellt ✅


In [7]:
frage = "Wie viele Mitarbeiter hat LAIKA?"

# Sucht die 2 ähnlichsten Chunks
relevante_docs = vectorstore.similarity_search(frage, k=2)

relevante_docs

[Document(metadata={}, page_content='Das Büro ist in Berlin. Wir haben 42 Mitarbeiter. Hunde sind im Büro erlaubt.'),
 Document(metadata={}, page_content='LAIKA wurde 2024 gegründet. CEO ist Anna Müller. Hauptprodukt ist ein KI-Assistent für Firmen.')]

In [8]:
print(f"Frage: {frage}\n")
print("Gefundene Chunks:")
for i, doc in enumerate(relevante_docs):
    print(f"  [{i+1}] {doc.page_content}")

Frage: Wie viele Mitarbeiter hat LAIKA?

Gefundene Chunks:
  [1] Das Büro ist in Berlin. Wir haben 42 Mitarbeiter. Hunde sind im Büro erlaubt.
  [2] LAIKA wurde 2024 gegründet. CEO ist Anna Müller. Hauptprodukt ist ein KI-Assistent für Firmen.


In [10]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Retrieval
chunks = vectorstore.similarity_search(frage, k=2)
kontext = "\n".join([d.page_content for d in chunks])
kontext

'Das Büro ist in Berlin. Wir haben 42 Mitarbeiter. Hunde sind im Büro erlaubt.\nLAIKA wurde 2024 gegründet. CEO ist Anna Müller. Hauptprodukt ist ein KI-Assistent für Firmen.'

In [11]:
# Prompt zusammenbauen
prompt = f"""Du bist ein Firmen-Assistent. Antworte NUR basierend auf dem Kontext.
Wenn die Antwort nicht im Kontext steht, sag "Weiß ich nicht".

Kontext:
{kontext}

Frage: {frage}"""

response = client.chat.completions.create(
    model="qwen2.5:14b",
    messages=[{"role": "user", "content": prompt}]
)

print(f"Antwort: {response.choices[0].message.content}")

Antwort: Die Frage zum Mitarbeiterbestand von LAIKA wurde in den gegebenen Informationen mit "Wir haben 42 Mitarbeiter" beantwortet, was implizit auf LAIKA hinweisen könnte. Daher lautet die Antwort:

LAIKA hat 42 Mitarbeiter.
